# VPA Demo: Workflow For Generation, Prediction, QC, And Ranking

Use this notebook when several tools should be connected automatically.
The workflow below combines RFdiffusion backbone generation,
ProteinMPNN sequence design, Protenix-v2 structure prediction,
structure analysis, protein QC, and ranking.

By default, this notebook is smoke-safe: it checks GPU/CLI visibility,
writes the example YAML to `/tmp`, and prints the heavy run command.
Set `VPA_DEMO_RUN_HEAVY=1` before executing the notebook to actually
start workers and run RFdiffusion/ProteinMPNN/Protenix-v2.

The workflow intentionally uses an internal conversion helper between
ProteinMPNN and Protenix-v2. That helper is fine inside an authored
workflow, but it is not exposed as a first-class LLM-agent tool.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import textwrap
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "vibeproteinagent").exists():
            return candidate
    return Path("/opt/vpa-data/VibeProteinAgent")

REPO_ROOT = find_repo_root()
VPA = os.environ.get("VPA_BIN", str(Path("/opt/miniconda3/envs/vpa/bin/vpa")))
if not Path(VPA).exists():
    VPA = os.environ.get("VPA_BIN", "vpa")

RUN_HEAVY = os.environ.get("VPA_DEMO_RUN_HEAVY", "0") == "1"
RUN_LLM = os.environ.get("VPA_DEMO_RUN_LLM", "0") == "1"
RUN_INSTALL = os.environ.get("VPA_DEMO_RUN_INSTALL", "0") == "1"
WORKSPACE_PREFIX = os.environ.get("VPA_DEMO_WORKSPACE_PREFIX", "demo")

print(f"repo: {REPO_ROOT}")
print(f"vpa: {VPA}")
print(f"RUN_INSTALL={RUN_INSTALL} RUN_HEAVY={RUN_HEAVY} RUN_LLM={RUN_LLM}")


def run(cmd, *, check=True, cwd=REPO_ROOT, env=None, timeout=None):
    if isinstance(cmd, str):
        printable = cmd
        shell = True
    else:
        printable = " ".join(shlex.quote(str(part)) for part in cmd)
        shell = False
    print(f"$ {printable}")
    proc = subprocess.run(
        cmd,
        shell=shell,
        cwd=str(cwd),
        env={**os.environ, **(env or {})},
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout[-6000:])
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {printable}")
    return proc


def show_command(parts):
    print(" ".join(shlex.quote(str(part)) for part in parts))


## Optional Install For This Workflow

Run this cell only when the workflow tools are not installed yet. The workflow below does not use ESM3; if you switch to `--profile full` or add ESM3, export `HF_TOKEN=<huggingface-access-token>` first when the model cache is not already present.


In [ ]:
install_cmd = [
    "npx", "vibeproteinagent", "install",
    "--tools", "rfdiffusion,proteinmpnn,protenix_v2,structure_analysis,protein_qc,rank_designs",
    "--cuda-profile", "auto",
    "--yes",
]
if RUN_INSTALL:
    run(install_cmd, timeout=10800)
else:
    show_command(install_cmd)

print("Optional ESM3/full-profile credential placeholder, not a real value:")
show_command(["bash", "-lc", "export HF_TOKEN=<huggingface-access-token>"])


## 1. GPU And Tool Readiness Smoke

In [ ]:
# This cell is intentionally lightweight. It proves the notebook is running on a
# GPU-capable node and that PyTorch can see CUDA. The heavy VPA jobs below are
# gated behind RUN_HEAVY so the tutorial remains safe to execute.
run("hostname")
run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | head -3", check=False)
import sys
run([sys.executable, "-c", "import torch; print('torch', torch.__version__); print('cuda_available', torch.cuda.is_available()); print('device_count', torch.cuda.device_count()); print('device0', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"])

In [ ]:
for tool in ["rfdiffusion", "proteinmpnn", "protenix_v2", "structure_analysis", "protein_qc", "rank_designs"]:
    run([VPA, "tool", "info", tool, "--json"], check=False, timeout=90)

## 2. Write The Workflow YAML

In [ ]:
demo_out_dir = Path(os.environ.get("VPA_DEMO_OUT_DIR", "/tmp")) / f"vpa_demo_{WORKSPACE_PREFIX}"
demo_out_dir.mkdir(parents=True, exist_ok=True)
workflow_path = demo_out_dir / "generated_rfdiffusion_mpnn_protenix_workflow.yaml"
workflow_path.write_text("""
name: demo_rfdiffusion_mpnn_protenix
version: "1.0.0"
description: "Generate a backbone, design sequences, refold, analyze, QC, and rank."
params:
  length: {type: int, default: 60}
  mpnn_num_designs: {type: int, default: 2}
  top_k: {type: int, default: 1}
nodes:
  - id: backbone
    tool: rfdiffusion
    inputs:
      mode: unconditional
      length_min: ${params.length}
      length_max: ${params.length}
      num_designs: 1
      num_diffusion_steps: 15
      seed: 101
    cache: false
  - id: design
    tool: proteinmpnn
    inputs:
      pdb_path: ${outputs.backbone.designs[0].pdb_path}
      num_designs: ${params.mpnn_num_designs}
      temperature: 0.1
      seed: 42
    cache: true
  - id: protenix_input
    tool: mpnn_to_protenix_input
    inputs:
      designs_fasta: ${outputs.design.designs_fasta}
      top_k: ${params.top_k}
      name_prefix: demo_refold
    cache: true
  - id: refold
    tool: protenix_v2
    foreach: ${outputs.protenix_input.input_jsons}
    max_parallel: 1
    inputs:
      input_json: ${item}
      seeds: [101]
      use_msa: false
      n_sample: 1
      n_step: 20
      n_cycle: 1
    cache: true
  - id: analyze_refold
    tool: structure_analysis
    foreach: ${outputs.refold}
    inputs:
      structure_path: ${item.predicted_cif}
      max_annotations: 100
    cache: true
  - id: qc
    tool: protein_qc
    foreach: ${outputs.refold}
    inputs:
      pdb_path: ${item.predicted_cif}
    cache: true
  - id: rank
    tool: rank_designs
    inputs:
      predictions: ${outputs.refold}
      analyses: ${outputs.analyze_refold}
      qc_results: ${outputs.qc}
      top_k: ${params.top_k}
    cache: true
""".strip() + "\\n")
print(workflow_path)
print(workflow_path.read_text()[:2000])

## 3. Inspect And Run The Workflow

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_workflow"
run([VPA, "workflow", "inspect", str(workflow_path)], check=False, timeout=60)
run_cmd = [
    VPA, "workflow", "run", str(workflow_path),
    "--workspace", workspace,
    "--param", "length=60",
    "--param", "mpnn_num_designs=2",
    "--param", "top_k=1",
    "--max-wait-s", "7200",
    "--follow",
]
if RUN_HEAVY:
    run([VPA, "daemon", "start", "--workers", "1", "--cpu-workers", "2", "--workspace", workspace], timeout=120)
    run(run_cmd, timeout=7800)
else:
    show_command([VPA, "daemon", "start", "--workers", "1", "--cpu-workers", "2", "--workspace", workspace])
    show_command(run_cmd)

## 4. Monitor, Resume, And Report

In [ ]:
workspace = f"{WORKSPACE_PREFIX}_workflow"
for parts in [
    [VPA, "workflow", "status", "<workflow_id>", "--workspace", workspace],
    [VPA, "workflow", "run", str(workflow_path), "--workspace", workspace, "--resume", "<workflow_id>"],
    [VPA, "report", "<workflow_id>", "--workspace", workspace, "--out", "demo_workflow_report.html", "--open"],
]:
    show_command(parts)